# Lesson 01 - AIエージェントの紹介

<strong>AIエージェント初心者向け</strong>コースの最初のレッスンへようこそ！

<strong>AIエージェント</strong>とは、大規模言語モデル（LLM）を推論エンジンとして使用し、ユーザーの代わりに目標を達成するために、APIの呼び出し、データベースの照会、コードの実行など、現実の世界で<em>アクション</em>を実行できるプログラムです。

このノートブックでは、初めてのエージェント、つまり休暇先をおすすめする<strong>トラベルエージェント</strong>を構築します。その過程で次のことを学びます：

1. <strong>Microsoft Agent Framework</strong>を使用してAzure AI Foundry Agent Serviceに接続する方法。
2. エージェントに呼び出せるプレーンなPython関数である<strong>ツール</strong>を与える方法。
3. エージェントを実行してその応答を検査する方法。
4. エージェントの応答をトークンごとにストリーミングする方法。


## セットアップ

このノートブックを実行する前に、以下を確認してください。

1. **デプロイされたチャットモデル（例：`gpt-4o-mini`）を持つ Azure AI Foundry プロジェクト** があること。
2. **Azure CLI にログインしていること** — ターミナルで `az login` を実行してください。
3. **必要な環境変数を設定していること：**
   - `AZURE_AI_PROJECT_ENDPOINT` — あなたの Azure AI Foundry プロジェクトのエンドポイント。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — デプロイされたモデルの名前。

以下のセルは必要な Python パッケージをインストールします。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 最初のエージェントを作成する

エージェントには次の2つが必要です:

- エージェントが<em>自分が誰であるか</em>と<em>どのように振る舞うか</em>を伝える<strong>指示</strong>（システムプロンプト）。
- エージェントが情報を取得したりアクションを実行したりするために呼び出せる、`@tool` デコレーターが付いた Python 関数という<strong>ツール</strong>。

以下では、人気の旅行先のリストを返す簡単なツールを定義します。ユーザーが旅行のおすすめを尋ねたときに、エージェントはこのツールを使用します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ストリーミングレスポンス

よりインタラクティブな体験のために、エージェントのレスポンスを<strong>ストリーミング</strong>できます。完全な応答を待つ代わりに、エージェントは生成されたテキストのチャンクを逐次返します。これは、リアルタイムで出力を表示したいチャットインターフェースで特に有用です。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

このレッスンでは以下のことを学びました：

- `AzureAIProjectAgentProvider` を使って Azure AI Foundry Agent Service に接続する <strong>プロバイダーを作成</strong> する方法。
- エージェントがあなたの Python 関数を呼び出せるように、`@tool` デコレーターを使って <strong>ツールを定義</strong> する方法。
- ユーザーメッセージで <strong>エージェントを実行</strong> し、その応答を表示する方法。
- <strong>リアルタイム出力のための応答ストリーミング</strong> の方法。

次のレッスンでは、エージェントフレームワークをより深く探求し、エージェントにより強力なツールと複数ステップの推論能力を与える方法を学びます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
